# 🎤 DSM-ASR: Mimi + Qwen3-0.6B

**Audio Prefix → Text Generation** ASR model:
- **Mimi** audio tokenizer encodes speech into discrete tokens
- **Qwen3-0.6B-Base** sees audio tokens as prefix, generates text
- **No timestamps needed** — model learns alignment itself

---
**Runtime**: A100 40GB

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone your repo
REPO_URL = "https://github.com/AhPro7/DSM.git"

!git clone {REPO_URL} /content/DSM
%cd /content/DSM

In [ ]:
# Install dependencies
!pip install -q torch transformers>=4.51.0 datasets[audio] accelerate \
    soundfile librosa tqdm jiwer wandb

In [ ]:
# Login to HuggingFace (for your dataset)
from huggingface_hub import login
login()  # Or: login(token="hf_YOUR_TOKEN")

## 2. Preprocess Dataset
Encodes all audio with Mimi. No timestamps needed!

In [ ]:
# Test with small batch first
!python data/prepare_data.py --max_samples 20

In [ ]:
# Check results
import json
with open("preprocessed_data/manifest.json") as f:
    m = json.load(f)
print(f"Processed: {m['total_processed']}, Errors: {m['total_errors']}")
if m['samples']:
    s = m['samples'][0]
    print(f"Sample: {s['num_frames']} frames, {s['duration']:.1f}s")
    print(f"Text: {s['text'][:100]}")

In [ ]:
# Full dataset (uncomment when ready)
# !python data/prepare_data.py

## 3. Sanity Checks

In [ ]:
!python data/dataset.py

In [ ]:
!python model/dsm_asr.py

## 4. Train
**Phase 1** (epochs 1-2): Frozen backbone → train audio embeddings

**Phase 2** (epochs 3+): Full fine-tuning

In [ ]:
# Quick test
!python train.py --max_steps 10 --batch_size 2 --max_samples 20

In [ ]:
# Full training
!python train.py --batch_size 4 --num_epochs 10

In [ ]:
# Plot training loss
import json, matplotlib.pyplot as plt
try:
    with open("output/training_log.json") as f:
        log = json.load(f)
    plt.figure(figsize=(10, 4))
    plt.plot([e['step'] for e in log], [e['loss'] for e in log])
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss')
    plt.grid(True, alpha=0.3); plt.show()
except: print('No log yet')

## 5. Evaluate

In [ ]:
!python evaluate.py --checkpoint output/best --output output/eval_results.json

## 6. Inference

In [ ]:
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
!python inference.py --checkpoint output/best --audio {audio_file}

## 7. Save Model

In [ ]:
# Save to HuggingFace Hub
from huggingface_hub import HfApi
api = HfApi()
repo_id = "nadsoft/dsm-asr-arabic"  # Change this
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="output/best", repo_id=repo_id)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")

In [ ]:
# Backup to Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r output/best /content/drive/MyDrive/dsm_asr_checkpoint/
print('✅ Saved to Drive')